# Road Sign Classifier Inference on PYNQ Z2

This notebook demonstrates how to run inference using the FINN-accelerated road sign classifier on the PYNQ Z2 board.

## Overview
1. Set up the PYNQ environment
2. Load the accelerator bitstream
3. Prepare input images
4. Run inference
5. Visualize and interpret results

## 1. Import Required Libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
import time
from pynq import Overlay
from pynq import allocate
import ipywidgets as widgets
from IPython.display import display, clear_output
import json
import cv2

# Check PYNQ version
import pynq
print(f"PYNQ version: {pynq.__version__}")

ModuleNotFoundError: No module named 'pynq'

## 2. Define Road Sign Classes

The German Traffic Sign Recognition Benchmark (GTSRB) dataset has 43 classes. Let's define them for interpretation of results.

In [ ]:
# Define class names for GTSRB dataset
class_names = {
    0: 'Speed limit (20km/h)',
    1: 'Speed limit (30km/h)',
    2: 'Speed limit (50km/h)',
    3: 'Speed limit (60km/h)',
    4: 'Speed limit (70km/h)',
    5: 'Speed limit (80km/h)',
    6: 'End of speed limit (80km/h)',
    7: 'Speed limit (100km/h)',
    8: 'Speed limit (120km/h)',
    9: 'No passing',
    10: 'No passing for vehicles over 3.5 metric tons',
    11: 'Right-of-way at the next intersection',
    12: 'Priority road',
    13: 'Yield',
    14: 'Stop',
    15: 'No vehicles',
    16: 'Vehicles over 3.5 metric tons prohibited',
    17: 'No entry',
    18: 'General caution',
    19: 'Dangerous curve to the left',
    20: 'Dangerous curve to the right',
    21: 'Double curve',
    22: 'Bumpy road',
    23: 'Slippery road',
    24: 'Road narrows on the right',
    25: 'Road work',
    26: 'Traffic signals',
    27: 'Pedestrians',
    28: 'Children crossing',
    29: 'Bicycles crossing',
    30: 'Beware of ice/snow',
    31: 'Wild animals crossing',
    32: 'End of all speed and passing limits',
    33: 'Turn right ahead',
    34: 'Turn left ahead',
    35: 'Ahead only',
    36: 'Go straight or right',
    37: 'Go straight or left',
    38: 'Keep right',
    39: 'Keep left',
    40: 'Roundabout mandatory',
    41: 'End of no passing',
    42: 'End of no passing by vehicles over 3.5 metric tons'
}

print(f"Total number of classes: {len(class_names)}")

## 3. Load Bitstream and Hardware Overlay

Load the FINN-generated bitstream (.bit) and hardware handoff file (.hwh) to configure the FPGA.

In [ ]:
# Set paths to bitstream and hwh files
# Update these paths to match your file locations
BITSTREAM_PATH = "road_sign_classifier.bit"
HWH_PATH = "road_sign_classifier.hwh"

# Check if files exist
if not os.path.exists(BITSTREAM_PATH):
    print(f"Error: Bitstream file not found at {BITSTREAM_PATH}")
    print("Please upload the .bit file to the PYNQ board and update the path.")
if not os.path.exists(HWH_PATH):
    print(f"Error: HWH file not found at {HWH_PATH}")
    print("Please upload the .hwh file to the PYNQ board and update the path.")

# Load the overlay
try:
    overlay = Overlay(BITSTREAM_PATH)
    print("✅ Bitstream loaded successfully!")
    print(f"Available IP blocks: {list(overlay.ip_dict.keys())}")
except Exception as e:
    print(f"Error loading bitstream: {e}")
    print("Please check the file paths and ensure the files are correctly uploaded.")

## 4. Define Image Preprocessing Functions

The FINN accelerator expects images in a specific format (32x32 RGB, normalized). Let's create functions to preprocess input images.

In [ ]:
def preprocess_image(image_path, target_size=(32, 32)):
    """Preprocess an image for the FINN accelerator."""
    # Load image
    img = Image.open(image_path)
    
    # Convert to RGB if needed
    if img.mode != 'RGB':
        img = img.convert('RGB')
    
    # Resize to target size
    img = img.resize(target_size, Image.LANCZOS)
    
    # Convert to numpy array and normalize to [0, 1]
    img_array = np.array(img).astype(np.float32) / 255.0
    
    # Reshape to NCHW format (batch_size=1, channels=3, height=32, width=32)
    # FINN expects this format
    img_nchw = img_array.transpose(2, 0, 1).reshape(1, 3, 32, 32)
    
    return img, img_array, img_nchw

def display_image_with_prediction(img, pred_class, pred_prob=None, top_k=3):
    """Display image with prediction results."""
    plt.figure(figsize=(8, 4))
    
    # Display image
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.axis('off')
    plt.title('Input Image')
    
    # Display prediction
    plt.subplot(1, 2, 2)
    
    if pred_prob is not None and top_k > 1:
        # Get top-k predictions
        top_indices = np.argsort(pred_prob[0])[-top_k:][::-1]
        top_probs = [pred_prob[0][i] for i in top_indices]
        
        # Create horizontal bar chart
        y_pos = np.arange(len(top_indices))
        plt.barh(y_pos, top_probs, align='center')
        plt.yticks(y_pos, [class_names[i] for i in top_indices])
        plt.xlabel('Probability')
        plt.title('Top Predictions')
    else:
        # Just show the top prediction
        plt.text(0.5, 0.5, f"Prediction: {class_names[pred_class]}", 
                 ha='center', va='center', fontsize=12)
        plt.axis('off')
        plt.title('Classification Result')
    
    plt.tight_layout()
    plt.show()

## 5. Set Up FINN Driver

Configure the FINN driver to interact with the accelerator.

In [ ]:
# Find the FINN driver name in the overlay
driver_name = None
for key in overlay.ip_dict.keys():
    if "finn" in key.lower() or "dataflow" in key.lower():
        driver_name = key
        break

if driver_name is None:
    print("Could not find FINN driver in the overlay.")
    print("Available IP blocks:")
    for key in overlay.ip_dict.keys():
        print(f"  - {key}")
    print("\nPlease manually set the driver_name variable to the correct IP block.")
    driver_name = "finn_design_0"  # Default name, may need to be changed
else:
    print(f"Found FINN driver: {driver_name}")

# Get the driver
try:
    finn_driver = getattr(overlay, driver_name)
    print("✅ FINN driver loaded successfully!")
except Exception as e:
    print(f"Error accessing FINN driver: {e}")
    print("Please check the driver name and ensure the bitstream is correctly loaded.")

## 6. Run Inference on Sample Images

Let's run inference on some sample road sign images.

In [ ]:
def run_inference(image_path):
    """Run inference on a single image using the FINN accelerator."""
    # Preprocess image
    img, img_array, img_nchw = preprocess_image(image_path)
    
    # Allocate input and output buffers
    input_size = 1 * 3 * 32 * 32  # NCHW format
    output_size = 1 * 43  # Batch size 1, 43 classes
    
    inp_buffer = allocate(shape=(input_size,), dtype=np.float32)
    out_buffer = allocate(shape=(output_size,), dtype=np.float32)
    
    # Copy input data to buffer
    inp_buffer[:] = img_nchw.flatten()
    
    # Run inference and measure time
    start_time = time.time()
    finn_driver.call(inp_buffer, out_buffer)
    inference_time = (time.time() - start_time) * 1000  # ms
    
    # Reshape output to [1, 43]
    output = out_buffer.reshape(1, 43)
    
    # Get prediction
    pred_class = np.argmax(output)
    pred_prob = output
    
    # Apply softmax to get probabilities
    exp_output = np.exp(output - np.max(output))
    softmax_output = exp_output / np.sum(exp_output, axis=1, keepdims=True)
    
    # Display results
    print(f"Inference time: {inference_time:.2f} ms")
    print(f"Predicted class: {pred_class} - {class_names[pred_class]}")
    print(f"Confidence: {softmax_output[0][pred_class]*100:.2f}%")
    
    # Display image with prediction
    display_image_with_prediction(img_array, pred_class, softmax_output, top_k=5)
    
    return pred_class, softmax_output

## 7. Test with Sample Images

Let's test the accelerator with some sample road sign images. You can upload your own images to the PYNQ board and test them.

In [ ]:
# Test with a sample image
# Replace with your own image path
sample_image_path = "stop_sign.jpg"

# Check if file exists
if not os.path.exists(sample_image_path):
    print(f"Error: Sample image not found at {sample_image_path}")
    print("Please upload an image to the PYNQ board and update the path.")
else:
    # Run inference
    pred_class, pred_prob = run_inference(sample_image_path)

## 8. Interactive Image Upload and Inference

Let's create an interactive widget to upload images and run inference.

In [ ]:
def process_upload(change):
    """Process uploaded image and run inference."""
    # Clear previous output
    clear_output(wait=True)
    
    # Get uploaded file content
    uploaded_file = change.new[0]
    content = uploaded_file['content']
    filename = uploaded_file['name']
    
    # Save uploaded file
    with open(filename, 'wb') as f:
        f.write(content)
    
    print(f"Uploaded: {filename}")
    print("Running inference...")
    
    # Run inference
    try:
        pred_class, pred_prob = run_inference(filename)
        print(f"\nTop 3 predictions:")
        top_indices = np.argsort(pred_prob[0])[-3:][::-1]
        for i, idx in enumerate(top_indices):
            print(f"{i+1}. Class {idx}: {class_names[idx]} - {pred_prob[0][idx]*100:.2f}%")
    except Exception as e:
        print(f"Error running inference: {e}")
    
    # Show upload widget again
    display(uploader)

# Create upload widget
uploader = widgets.FileUpload(
    accept='image/*',  # Accept only images
    multiple=False,    # Only one file at a time
    description='Upload Image:'
)

# Register callback
uploader.observe(process_upload, names='value')

# Display widget
display(uploader)

## 9. Webcam Capture (Optional)

If your PYNQ board has a connected webcam, you can use it to capture images and run inference.

In [ ]:
def capture_and_classify():
    """Capture image from webcam and run inference."""
    try:
        # Initialize webcam
        cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            print("Error: Could not open webcam.")
            return
        
        # Capture frame
        ret, frame = cap.read()
        if not ret:
            print("Error: Could not capture frame.")
            return
        
        # Release webcam
        cap.release()
        
        # Save frame
        cv2.imwrite("webcam_capture.jpg", frame)
        
        # Convert BGR to RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Display captured image
        plt.figure(figsize=(8, 6))
        plt.imshow(frame_rgb)
        plt.title("Captured Image")
        plt.axis('off')
        plt.show()
        
        # Run inference
        print("Running inference...")
        pred_class, pred_prob = run_inference("webcam_capture.jpg")
        
    except Exception as e:
        print(f"Error: {e}")

# Create button widget
capture_button = widgets.Button(
    description='Capture & Classify',
    button_style='success',
    icon='camera'
)

# Register callback
capture_button.on_click(lambda b: capture_and_classify())

# Display button
display(widgets.HTML("<b>Webcam Capture (if available):</b>"))
display(capture_button)

## 10. Performance Benchmarking

Let's benchmark the inference performance of the FINN accelerator.

In [ ]:
def benchmark_inference(image_path, num_runs=100):
    """Benchmark inference performance."""
    # Preprocess image
    _, _, img_nchw = preprocess_image(image_path)
    
    # Allocate input and output buffers
    input_size = 1 * 3 * 32 * 32  # NCHW format
    output_size = 1 * 43  # Batch size 1, 43 classes
    
    inp_buffer = allocate(shape=(input_size,), dtype=np.float32)
    out_buffer = allocate(shape=(output_size,), dtype=np.float32)
    
    # Copy input data to buffer
    inp_buffer[:] = img_nchw.flatten()
    
    # Warm-up run
    finn_driver.call(inp_buffer, out_buffer)
    
    # Benchmark runs
    inference_times = []
    for _ in range(num_runs):
        start_time = time.time()
        finn_driver.call(inp_buffer, out_buffer)
        inference_time = (time.time() - start_time) * 1000  # ms
        inference_times.append(inference_time)
    
    # Calculate statistics
    avg_time = np.mean(inference_times)
    min_time = np.min(inference_times)
    max_time = np.max(inference_times)
    std_time = np.std(inference_times)
    fps = 1000 / avg_time
    
    # Print results
    print(f"Benchmark results ({num_runs} runs):")
    print(f"  Average inference time: {avg_time:.2f} ms")
    print(f"  Min inference time: {min_time:.2f} ms")
    print(f"  Max inference time: {max_time:.2f} ms")
    print(f"  Standard deviation: {std_time:.2f} ms")
    print(f"  Throughput: {fps:.2f} FPS")
    
    # Plot histogram
    plt.figure(figsize=(10, 6))
    plt.hist(inference_times, bins=20)
    plt.axvline(avg_time, color='r', linestyle='dashed', linewidth=2, label=f'Mean: {avg_time:.2f} ms')
    plt.xlabel('Inference Time (ms)')
    plt.ylabel('Frequency')
    plt.title('Inference Time Distribution')
    plt.legend()
    plt.grid(True)
    plt.show()
    
    return avg_time, fps

# Run benchmark if sample image exists
if os.path.exists(sample_image_path):
    avg_time, fps = benchmark_inference(sample_image_path, num_runs=50)
else:
    print(f"Error: Sample image not found at {sample_image_path}")
    print("Please upload an image to the PYNQ board and update the path.")

## 11. Conclusion

Congratulations! You have successfully deployed and tested your FINN-accelerated road sign classifier on the PYNQ Z2 board.

### Summary
- Loaded the FINN-generated bitstream and hardware overlay
- Set up the FINN driver to interact with the accelerator
- Preprocessed images for inference
- Ran inference on sample images
- Created an interactive interface for image upload and classification
- Benchmarked the inference performance

### Next Steps
- Try with more road sign images
- Integrate with a camera for real-time classification
- Optimize the model further for better performance
- Explore other FINN features and capabilities